# Phase 6 — The Walk-Forward Test (the headline result)

**The idea in one line.** So far we picked pairs *once* (on 2015–2019) and traded them for six years.
Real trading doesn't work like that — you re-check your pairs regularly. This notebook does exactly
that: every quarter it re-groups the stocks, re-finds the tradeable pairs, and trades them for the
next three months, chaining 27 quarters into one continuous track record.

**Why this is the headline result.** The Phase 5 backtest answered "did the 2019 pair list keep
working?" This one answers the much harder, more realistic question: "if we had *run this system live*
quarter after quarter — always choosing pairs using only information available at the time — would it
have made money?" That is the number the dissertation stands on.

**What stays fixed, and what refreshes, at every quarter:**

| Component | Fixed or refreshed? | Why |
|---|---|---|
| VAE (the fingerprint-maker) | **Frozen** — trained once on 2015–2017, never touched again | Keeps the fingerprint space a fixed ruler; the honest, look-ahead-free design from Phase 2 |
| Behaviour groups (HDBSCAN) | **Refreshed** each quarter on the latest fingerprints | Relationships drift; this is the "adapts to changing markets" part of the project aim |
| Pair tests + filters (Engle–Granger) | **Refreshed** each quarter on the latest prices | A pair must re-prove its springiness on recent data before we trade it again |
| Trading rule (z-score entry/exit) | **Fixed** — identical to Phase 5 | One rule everywhere; no tuning along the way |

**The calendar.** Our quarters are simply the 60-trading-day windows built in Phase 1 (windows 20–46).
At the start of window *w* we form pairs using only windows *w−20 … w−1* (≈ the previous five years),
then trade them across window *w*. First traded day 2019-10-10, last 2026-03-23 — 27 quarters. (The
final 18 trading days of the dataset fall beyond the last complete window and are left untraded.)

## The design decisions (each one a viva question)

- **Formation = the last 20 windows (≈ 5 years), rolling.** The static pipeline formed pairs on
  windows 0–19; using the same-length lookback at every rebalance means each quarter's selection is
  built exactly the way Phases 3–4 were — just slid forward. Result: the walk-forward and the static
  run differ only in *when* selection happens, not *how*.
- **"Re-encoding" costs nothing here.** Phase 2 already pushed all 47 windows of every stock through
  the frozen VAE and saved the fingerprints. Re-encoding at rebalance *w* is therefore just *slicing*
  the saved fingerprints for windows *w−20 … w−1* — the frozen model would produce identical numbers,
  so nothing is recomputed and nothing can leak.
- **Grouping settings frozen from Phase 3** (smallest group 3, tightness 3, "pick the tight clumps"
  mode). Those settings were chosen on pre-2020 data; re-tuning them each quarter would be a hidden
  form of peeking. Same for the two blank fingerprint numbers we exclude: which numbers are blank is a
  property of the frozen VAE, decided once on the pre-2020 formation data.
- **A lighter version of the Phase 3 stability check runs every quarter** (50 re-groupings on random
  90% subsets; a group must keep its members together at least half the time). Same guard, same
  threshold — just fewer repeats so 27 rebalances stay fast.
- **Pair filters and trading rule are byte-for-byte the Phase 4 / Phase 5 ones.** Springiness p < 0.05,
  spring-back speed 5–60 days, gap size ≥ 1%, then the ten strongest pairs with no stock reused, each
  trading the ±2 / ±0.5 z-score rule with yesterday's signal.
- **Money is kept in ten fixed 10% sleeves.** Each selected pair runs in its own sleeve; if a quarter
  yields only six pairs, four sleeves simply sit in cash. This keeps risk per pair constant through
  time (the alternative — splitting everything across however many pairs exist — would quietly double
  each bet in thin quarters).
- **All positions are closed on the last day of each quarter.** The next quarter may pick different
  pairs, so we settle up cleanly at each rebalance. The extra trading this causes is exactly what the
  Phase 8 cost model will charge for — realism, priced.

## Why the engine lives in a file (`walkforward.py`)

The next cell doesn't run anything — it *writes* the engine to `walkforward.py` (the `%%writefile`
line at the top does that), and the cell after imports it back. The reason is Phase 9: the benchmark
only proves something if the GARCH and correlation variants run through the **identical machinery**
with only the fingerprint swapped. By importing this one file everywhere, any difference in results
can only come from the encoder — never from a quietly different backtest. The full code stays visible
right here in the notebook.

In [ ]:
%%writefile walkforward.py
'''Walk-forward engine, shared by Phase 6 (VAE) and Phase 9 (GARCH / correlation benchmarks).

Every function is the Phase 4 / Phase 5 logic unchanged, packaged so the encoder is swappable:
a selector receives (window number, formation log-prices, tickers) and returns the pairs to trade.
'''
import numpy as np
import pandas as pd
import hdbscan
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from itertools import combinations

# ---- pair testing (identical to Phase 4) -------------------------------------

def half_life(s):
    ds = np.diff(s)
    beta = np.polyfit(s[:-1], ds, 1)[0]
    return -np.log(2) / beta if beta < 0 else np.nan

def eg_pair(logp, a, b):
    y, x = logp[a].values, logp[b].values
    res = sm.OLS(y, sm.add_constant(x)).fit()
    return res.params[1], coint(y, x)[1], half_life(res.resid), res.resid.std()

def best_pair(logp, a, b):
    h1, p1, hl1, sd1 = eg_pair(logp, a, b)
    h2, p2, hl2, sd2 = eg_pair(logp, b, a)
    if p1 <= p2:
        return {'asset1': a, 'asset2': b, 'hedge_ratio': h1, 'eg_pvalue': p1, 'half_life': hl1, 'spread_std': sd1}
    return {'asset1': b, 'asset2': a, 'hedge_ratio': h2, 'eg_pvalue': p2, 'half_life': hl2, 'spread_std': sd2}

def johansen_ok(logp, a, b):
    j = coint_johansen(np.column_stack([logp[a].values, logp[b].values]), 0, 1)
    return bool(j.lr1[0] > j.cvt[0, 1])          # trace stat > 95% critical value

# ---- clustering with the per-rebalance stability guard -----------------------

def cluster_stable(F, mcs=3, ms=3, n_boot=50, keep=0.5, seed=0):
    '''HDBSCAN (leaf mode) + drop any cluster whose members do not stay together
    in at least `keep` of the bootstrap re-groupings (the Phase 3 guard, lighter).'''
    fit = lambda X: hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms, metric='euclidean',
                                    cluster_selection_method='leaf').fit_predict(X)
    labels = fit(F)
    n = len(F)
    co, cnt = np.zeros((n, n)), np.zeros((n, n))
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        idx = np.sort(rng.choice(n, int(0.9 * n), replace=False))
        lbl = fit(F[idx])
        cnt[np.ix_(idx, idx)] += 1
        for c in set(lbl) - {-1}:
            mem = idx[lbl == c]
            co[np.ix_(mem, mem)] += 1
    freq = np.divide(co, cnt, out=np.zeros_like(co), where=cnt > 0)
    out = labels.copy()
    for c in set(labels) - {-1}:
        mem = np.where(labels == c)[0]
        sub = freq[np.ix_(mem, mem)]
        if sub[np.triu_indices_from(sub, 1)].mean() < keep:
            out[mem] = -1
    return out

# ---- pair selection (Phase 4 gates + Phase 5 no-shared-legs top-N) -----------

P_MAX, HL_MIN, HL_MAX, SD_MIN, N_PAIRS = 0.05, 5, 60, 0.01, 10

def gate_and_pick(cands, logp_form):
    if not cands:
        return pd.DataFrame(columns=['asset1', 'asset2', 'hedge_ratio', 'eg_pvalue',
                                     'half_life', 'spread_std']), 0, np.nan
    df = pd.DataFrame(cands)
    n_tested = len(df)
    df = df[(df.eg_pvalue < P_MAX) & df.half_life.between(HL_MIN, HL_MAX)
            & (df.spread_std >= SD_MIN)].sort_values('eg_pvalue')
    used, picked = set(), []
    for r in df.itertuples():
        if r.asset1 in used or r.asset2 in used:
            continue
        picked.append(r.Index)
        used.update([r.asset1, r.asset2])
        if len(picked) == N_PAIRS:
            break
    sel = df.loc[picked].copy()
    agree = (np.mean([johansen_ok(logp_form, r.asset1, r.asset2) for r in sel.itertuples()])
             if len(sel) else np.nan)
    return sel, n_tested, agree

def cluster_selector(features_fn):
    '''Selector for the VAE and GARCH chains: fingerprint -> stable clusters -> test within.'''
    def select(w, logp_form, tickers):
        labels = cluster_stable(features_fn(w))
        cands = []
        for c in set(labels) - {-1}:
            mem = [tickers[i] for i in np.where(labels == c)[0]]
            cands += [best_pair(logp_form, a, b) for a, b in combinations(mem, 2)]
        sel, n_tested, agree = gate_and_pick(cands, logp_form)
        return sel, n_tested, agree, len(set(labels) - {-1})
    return select

def corr_selector(top_n=200):
    '''Classical baseline: the top-N most-correlated pairs in the whole universe, same gates after.'''
    def select(w, logp_form, tickers):
        corr = logp_form.diff().corr().values
        iu = np.triu_indices(len(tickers), 1)
        order = np.argsort(corr[iu])[::-1][:top_n]
        cands = [best_pair(logp_form, tickers[iu[0][k]], tickers[iu[1][k]]) for k in order]
        sel, n_tested, agree = gate_and_pick(cands, logp_form)
        return sel, n_tested, agree, 0
    return select

# ---- trading (identical to Phase 5) ------------------------------------------

def zscore(s, w=60):
    return (s - s.rolling(w).mean()) / s.rolling(w).std()

def positions(z, entry=2.0, exit_=0.5):
    pos, state = np.zeros(len(z)), 0
    for i, zi in enumerate(z.to_numpy()):
        if np.isnan(zi):
            state = 0
        elif state == 0:
            state = -1 if zi > entry else (1 if zi < -entry else 0)
        elif state == 1 and zi >= -exit_:
            state = 0
        elif state == -1 and zi <= exit_:
            state = 0
        pos[i] = state
    return pd.Series(pos, index=z.index)

def trade_quarter(logp, ret, sel, t0, t1):
    '''Trade the selected pairs across [t0, t1] in fixed 1/N_PAIRS sleeves.
    Returns daily portfolio return and daily turnover (fraction of capital traded);
    every position is force-closed at the quarter's last close.'''
    days = logp.loc[t0:t1].index
    pnl = pd.Series(0.0, index=days)
    tvr = pd.Series(0.0, index=days)
    for r in sel.itertuples():
        z = zscore(logp[r.asset1] - r.hedge_ratio * logp[r.asset2]).loc[t0:t1]
        eff = positions(z).shift(1).fillna(0)                  # act on yesterday's signal
        leg = (ret[r.asset1] - r.hedge_ratio * ret[r.asset2]).loc[t0:t1].fillna(0)
        pnl += eff * leg / (1 + abs(r.hedge_ratio)) / N_PAIRS
        dpos = eff.diff().abs().fillna(0)
        dpos.iloc[-1] += abs(eff.iloc[-1])                     # forced close at quarter end
        tvr += dpos / N_PAIRS
    return pnl, tvr

# ---- the walk-forward loop ---------------------------------------------------

def run_walk_forward(selector, logp, ret, windows, w_start=20, w_end=46, lookback=20, label='wf'):
    all_ret, all_tvr, pair_rows, summary = [], [], [], []
    for w in range(w_start, w_end + 1):
        f0, f1 = windows.start_date[w - lookback], windows.end_date[w - 1]
        t0, t1 = windows.start_date[w], windows.end_date[w]
        sel, n_tested, agree, n_clusters = selector(w, logp.loc[f0:f1], list(logp.columns))
        pnl, tvr = trade_quarter(logp, ret, sel, t0, t1)
        all_ret.append(pnl)
        all_tvr.append(tvr)
        for r in sel.itertuples():
            pair_rows.append({'window': w, 'asset1': r.asset1, 'asset2': r.asset2,
                              'hedge_ratio': r.hedge_ratio, 'eg_pvalue': r.eg_pvalue,
                              'half_life': r.half_life})
        q_ret = float((1 + pnl).prod() - 1)
        summary.append({'window': w, 'trade_start': t0, 'trade_end': t1, 'n_tested': n_tested,
                        'n_pairs': len(sel), 'n_clusters': n_clusters,
                        'johansen_agree': agree, 'quarter_return': q_ret})
        print(f'{label} w{w:02d} {t0} -> {t1}: tested {n_tested:4d}, traded {len(sel):2d}, quarter {q_ret:+7.2%}')
    return (pd.concat(all_ret), pd.concat(all_tvr),
            pd.DataFrame(pair_rows), pd.DataFrame(summary))

# ---- scorecard (identical to Phase 5) ----------------------------------------

def perf(pr):
    eq = (1 + pr).cumprod()
    cum = eq.iloc[-1] - 1
    ann = (1 + cum) ** (252 / len(pr)) - 1
    sharpe = pr.mean() / pr.std() * np.sqrt(252) if pr.std() > 0 else np.nan
    mdd = (eq / eq.cummax() - 1).min()
    return {'cum_return': cum, 'ann_return': ann, 'sharpe': sharpe, 'max_drawdown': mdd}, eq

## Load the inputs and define the VAE fingerprint

Only three inputs are needed — the saved fingerprints from the frozen VAE, the window calendar, and
the cleaned prices. The fingerprint recipe is exactly Phase 3's: for the last 20 windows, take each
active fingerprint number's **average** and **spread**, then put every column on the same footing.
The two blank ("collapsed") fingerprint numbers are excluded using the same rule as Phase 3, measured
once on the pre-2020 formation windows and frozen — which numbers are blank is a property of the
frozen VAE, not something we re-decide along the way.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from walkforward import cluster_selector, run_walk_forward, perf

Z = np.load('data/model_input/latent_vectors.npy')                    # (462, 47, 12)
tickers = pd.read_csv('data/model_input/tickers.csv', header=None)[0].tolist()
windows = pd.read_csv('data/model_input/window_dates.csv', index_col='window')
prices = pd.read_parquet('data/processed/adj_close_clean.parquet')[tickers]
logp = np.log(prices)
ret = prices.pct_change()

LOOKBACK = 20                                                          # 20 windows = ~5 years, as in Phase 3/4
dim_std = Z[:, :LOOKBACK, :].reshape(-1, Z.shape[-1]).std(0)
active = dim_std > 0.3 * np.median(dim_std)                            # same rule as Phase 3, frozen
print(f'Active fingerprint numbers: {active.sum()} / {len(active)}')

def vae_features(w):
    Zf = Z[:, w - LOOKBACK:w, :][:, :, active]
    F = np.concatenate([Zf.mean(1), Zf.std(1)], axis=1)
    return (F - F.mean(0)) / F.std(0)

print(f'Quarters to trade: windows 20-46 '
      f'({windows.start_date[20]} -> {windows.end_date[46]})')

## Run it

One line per quarter will print below: how many pairs were tested, how many were good enough to
trade, and what the quarter returned. Expect a few minutes' runtime (each quarter re-groups the
stocks 51 times for the stability guard and runs a few hundred springiness tests).

**Red flags to watch for:** many quarters trading 0–2 pairs (the selector is starving), cluster
counts collapsing to 1 (the grouping has degenerated), or every quarter losing money after 2022
(the signal has died). Any of these becomes a finding to report honestly, not to hide.

In [ ]:
wf_ret, wf_tvr, wf_pairs, wf_summary = run_walk_forward(
    cluster_selector(vae_features), logp, ret, windows, label='VAE')

m, eq = perf(wf_ret)
print()
print(pd.Series(m).round(4).to_string())
print(f'quarters traded: {len(wf_summary)}, avg pairs/quarter: {wf_summary.n_pairs.mean():.1f}, '
      f'avg Johansen agreement: {wf_summary.johansen_agree.mean():.0%}')

## The headline picture

**Top — the running pot.** £1 traded through all 27 quarters, every pair chosen with only
information available at the time. This line *is* the dissertation's headline result (before costs —
Phase 8 adds those). The grey dashed line marks £1: above it we're up, below it we're down.

**Bottom — the dips.** How far the pot is below its best level so far ("how uncomfortable was it to
hold"). Shallow dips = a strategy you could actually live with.

**Bottom — the rolling one-year Sharpe.** Reward-for-risk over a sliding 252-day window: does the
edge persist, fade, or come and go with regimes? A line mostly above zero is a durable edge; its
negative stretches locate exactly *when* the system struggled — the raw material for the
per-quarter attribution discussion.

For context we print the Phase 5 static portfolio's scorecard next to the walk-forward one. They are
not rivals — the static test held one fixed 2019 pair list, the walk-forward refreshes quarterly —
but the comparison shows what refreshing buys (or costs).

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True,
                       gridspec_kw={'height_ratios': [2, 1, 1]})
ax[0].plot(eq.index, eq.values, color='navy')
ax[0].axhline(1, color='grey', ls='--', lw=0.7)
ax[0].set_title('Walk-forward equity curve — frozen VAE, quarterly re-selection (gross of costs)')
dd = eq / eq.cummax() - 1
ax[1].fill_between(dd.index, dd.values, color='indianred')
ax[1].set_title('Drawdown')
roll = wf_ret.rolling(252)
ax[2].plot(eq.index, (roll.mean() / roll.std() * np.sqrt(252)).values, color='purple', lw=0.9)
ax[2].axhline(0, color='k', lw=0.5)
ax[2].set_title('Rolling one-year Sharpe')
plt.tight_layout(); plt.show()

static = pd.read_csv('data/model_input/strategy_metrics.csv', index_col=0)
side = pd.concat([pd.Series(m, name='walk_forward'), static['portfolio'].rename('static_portfolio')], axis=1)
print(side.round(4).to_string())

## Under the bonnet, quarter by quarter

- **Top — each quarter's return.** Green above zero, red below. We want most bars green and no single
  catastrophic red bar.
- **Bottom — how much raw material the selector had.** Pairs traded (out of a maximum of 10) and
  behaviour groups found, per quarter. A healthy system keeps finding groups and filling most sleeves;
  a starving one is a warning that pair relationships have thinned out — which itself becomes a
  Phase 7 talking point.

In [ ]:
q = wf_summary.set_index('window')
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].bar(q.index, q.quarter_return,
          color=['seagreen' if v > 0 else 'indianred' for v in q.quarter_return])
ax[0].axhline(0, color='k', lw=0.6)
ax[0].set_title('Return per quarter')
ax[1].plot(q.index, q.n_pairs, 'o-', label='pairs traded')
ax[1].plot(q.index, q.n_clusters, 's--', label='behaviour groups', alpha=0.7)
ax[1].set_ylim(0, None); ax[1].legend()
ax[1].set_title('Selector supply per quarter'); ax[1].set_xlabel('window')
plt.tight_layout(); plt.show()

## Save the hand-off files

- **wf_returns.csv** — the daily walk-forward returns (Phase 8 charges costs against these; Phase 9
  benchmarks against them).
- **wf_turnover.csv** — the fraction of capital traded each day (what Phase 8's cost model multiplies).
- **wf_pairs.csv** — which pairs were traded each quarter (Phase 7's persistence analysis reads this).
- **wf_summary.csv** — one row per quarter (supply, agreement, return — attribution for the write-up).
- **wf_metrics.csv** — the headline scorecard.

In [ ]:
wf_ret.rename('ret').to_csv('data/model_input/wf_returns.csv')
wf_tvr.rename('turnover').to_csv('data/model_input/wf_turnover.csv')
wf_pairs.to_csv('data/model_input/wf_pairs.csv', index=False)
wf_summary.to_csv('data/model_input/wf_summary.csv', index=False)
pd.Series(m).to_csv('data/model_input/wf_metrics.csv')
print('Saved wf_returns / wf_turnover / wf_pairs / wf_summary / wf_metrics')

## Summary

We ran the full pipeline the way it would run in real life: **27 consecutive quarters (Oct-2019 →
Mar-2026), each one re-grouping the stocks with the frozen VAE's fingerprints, re-testing pairs on
the latest five years, and trading the survivors for three months** — with the trading rule, filters,
and thresholds locked to their pre-2020 settings throughout. The result is one continuous, honest
out-of-sample equity curve, plus per-quarter attribution.

Phase 7 will ask how long the chosen pairs *live* across those rebalances; Phase 8 will charge
realistic trading costs against this curve; Phase 9 will re-run this identical engine with simpler
encoders to see whether the VAE earns its keep.